In [5]:
class Category:
    def __init__(self, name) :
        self.name   = name
        self.ledger = []

    def deposit(self, amount, description = "") :
        if not isinstance(amount, (int, float)) :
            raise TypeError("Amount must be a numerical value.")
            return False
        
        if amount <= 0 :
            raise ValueError("Amount must be a positive value.")
            return False

        self.ledger.append(
            {
                'amount' : amount,
                'description' : description
            }
        )

        return True


    def withdraw(self, amount, description = "") :
        if not isinstance(amount, (int, float)) :
            raise TypeError("Amount must be a numerical value.")
            return False
        
        if amount <= 0 :
            raise ValueError("Amount must be a positive value.")
            return False

        if not self.check_funds(amount) :
            return False

        self.ledger.append(
            {
                'amount' : -amount,
                'description' : description
            }
        )

        return True


    def get_balance(self) :
        if not self.ledger :
            return 0

        return sum([funds['amount'] for funds in self.ledger])


    def transfer(self, amount, category) :
        if not isinstance(amount, (int, float)) :
            raise TypeError("(Amount must be a numerical value.")
            return False
        
        if amount <= 0 :
            raise ValueError("Amount must be a positive value.")
            return False
        
        if not isinstance(category, Category) :
            raise TypeError("category must be an object of class Category")
            return False
        
        if not self.check_funds(amount) :
            return False

        result = []
        try :
            result.append(self.withdraw(amount, f"Transfer to {category.name}"))
            result.append(category.deposit(amount, f"Transfer from {self.name}"))
        except (ValueError, TypeError) :
            return False

        if not all(result) :
            return False

        return True


    def check_funds(self, amount) :
        available_funds = self.get_balance()
        if available_funds >= amount :
            return True

        return False


    def __str__(self) :
        # header
        line_length = 30 - len(self.name)
        show = f"{int(line_length / 2) * '*'}{self.name}{int(line_length - line_length / 2) * '*'}\n"

        # ledger and amount
        for item in self.ledger :
            line_length = 23 - len(item['description'])
            show += f"{item['description'][:23]}{line_length * ' '}"

            line_length = 7 - len(f"{(item['amount']):.2f}")
            show += f"{line_length * ' '}{(item['amount']):.2f}\n"

        # total
        show += f"Total: {round(self.get_balance(), 2)}"

        return show

In [ ]:
def create_spend_chart(categories):
    char    = 'o'
    chart   = "Percentage spent by category\n"
    x_axis  = [categ.name for categ in categories]
    y_axis  = [n for n in range(0, 110, 10)]
    y_axis.reverse()

    h_line  = f"    {len(x_axis) * 3 * '-'}--"
    max_index = max(range(len(x_axis)), key=lambda i: len(x_axis[i]))
    x_axis_legend = ""
    for idx in range(0, len(x_axis[max_index])) :
        if idx == 0 :
            x_axis_legend += "    "
        else :
            x_axis_legend += "\n    "
        for x in x_axis :
            if len(x) > idx :
                x_axis_legend += f" {x[idx]} "
            else :
                x_axis_legend += "   "


    withdrawals = []
    total       = []
    percentages = []

    for categ in categories :
        withdrawals.extend([abs(sum(item['amount'] for item in categ.ledger if item['amount'] < 0))])

    total       = sum(withdrawals)
    percentages = [int(withdrawals[index] / total * 100 / 10) * 10 for index in range(len(withdrawals))]

    for y in y_axis :
        if y == 100 :
            chart += f"\n{y}|"
        elif y >= 10 :
            chart += f"\n {y}|"
        else :
            chart += f"\n  {y}|"

        for categ_index, categ in enumerate(categories) :
            if percentages[categ_index] < y :
                chart += "   "
            else :
                chart += f" {char} "

    chart += '\n' + h_line + '\n' + x_axis_legend

    return chart

In [7]:
food = Category('Food')

try :
    food.deposit(1000, 'initial deposit')
    food.withdraw(10.15, 'groceries')
    food.withdraw(15.89, 'restaurant and more food for dessert')
    clothing = Category('Clothing')
    food.transfer(50, clothing)
except (ValueError, TypeError) as e :
    print("Error: ", e)
finally : 
    print(food)

*************Food*************
initial deposit        1000.00
groceries               -10.15
restaurant and more foo -15.89
Transfer to Clothing    -50.00
Total: 923.96


In [8]:
food     = Category('Food')
clothing = Category('Clothing')
auto     = Category('Auto')

food.deposit(1000, 'initial deposit')
food.withdraw(105.55, 'groceries')
food.withdraw(33.40, 'restaurant')
clothing.deposit(500, 'initial deposit')
clothing.withdraw(75.00, 'shirt')
auto.deposit(500, 'initial deposit')
auto.withdraw(25.00, 'fuel')

categories = [food, clothing, auto]

# Verify
result = create_spend_chart(categories)
print(repr(result))   # shows \n and exact spacing
print(result)         # visual check

TypeError: unsupported operand type(s) for +: 'int' and 'list'